# **Part 1: Deep Learning for NLP**

In this notebook, you will be given a basic implementation which uses a Convolutional Neural Network (CNN) for sentiment classification. The implementation is based on this original paper: [Convolutional Neural Networks for Sentence Classification](https://arxiv.org/pdf/1408.5882.pdf) we sent to you one week ago.

First, let's install the required libaries and run the given scripts to see how the performance looks like.




In [ ]:
!pip install datasets

Here we import relevant libraries and fix the random seed.

In [ ]:
import collections

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchtext
import tqdm

seed = 1234
device = "cuda"
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

Let's use the "MR" (movie review) dataset from the paper. The following script loads the dataset and prepare the training set, validation set and test set. The maximum sequence length is set to be 256, following the paper.

In [ ]:
dataset = "mr"

if dataset == "mr":
  train_data, valid_data, test_data = datasets.load_dataset("jeffnyman/rotten_tomatoes_reviews", split=["train", "validation", "test"])

if dataset == "subj":
  train_data, test_data = datasets.load_dataset("SetFit/subj", split=["train", "test"])
  train_valid_data = train_data.train_test_split(test_size=0.1)
  train_data = train_valid_data["train"]
  valid_data = train_valid_data["test"]

tokenizer = torchtext.data.utils.get_tokenizer("basic_english")
def tokenize_example(example, tokenizer, max_length):
    tokens = tokenizer(example["text"])[:max_length]
    return {"tokens": tokens}

max_length = 256

train_data = train_data.map(
    tokenize_example, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length}
)
valid_data = valid_data.map(
    tokenize_example, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length}
)
test_data = test_data.map(
    tokenize_example, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length}
)

Build a vocabulary based on the training data and convert the input tokens to indexes in the vocabulary.

In [ ]:
min_freq = 5
special_tokens = ["<unk>", "<pad>"]

vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

unk_index = vocab["<unk>"]
pad_index = vocab["<pad>"]
vocab.set_default_index(unk_index)

def numericalize_example(example, vocab):
    ids = vocab.lookup_indices(example["tokens"])
    return {"ids": ids}

train_data = train_data.map(numericalize_example, fn_kwargs={"vocab": vocab})
valid_data = valid_data.map(numericalize_example, fn_kwargs={"vocab": vocab})
test_data = test_data.map(numericalize_example, fn_kwargs={"vocab": vocab})

train_data = train_data.with_format(type="torch", columns=["ids", "label"])
valid_data = valid_data.with_format(type="torch", columns=["ids", "label"])
test_data = test_data.with_format(type="torch", columns=["ids", "label"])

Define a batch and data loader.

In [ ]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_ids = [i["ids"] for i in batch]
        batch_ids = nn.utils.rnn.pad_sequence(
            batch_ids, padding_value=pad_index, batch_first=True
        )
        batch_label = [i["label"] for i in batch]
        batch_label = torch.stack(batch_label)
        batch = {"ids": batch_ids, "label": batch_label}
        return batch

    return collate_fn

def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
  collate_fn = get_collate_fn(pad_index)
  data_loader = torch.utils.data.DataLoader(
      dataset=dataset,
      batch_size=batch_size,
      collate_fn=collate_fn,
      shuffle=shuffle,
  )
  return data_loader


The following script builds a CNN model according to the paper.

In [ ]:
class CNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        n_filters,
        filter_sizes,
        output_dim,
        pad_index,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_index)
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(embedding_dim, n_filters, filter_size)
                for filter_size in filter_sizes
            ]
        )
        self.fc = nn.Linear(len(filter_sizes) * n_filters, output_dim)


    def forward(self, ids):
        # ids = [batch size, seq len]
        embedded = self.embedding(ids)
        # embedded = [batch size, seq len, embedding dim]
        embedded = embedded.permute(0, 2, 1)
        # embedded = [batch size, embedding dim, seq len]
        conved = [torch.relu(conv(embedded)) for conv in self.convs]
        # conved_n = [batch size, n filters, seq len - filter_sizes[n] + 1]
        pooled = [conv.mean(dim=-1) for conv in conved]
        # pooled_n = [batch size, n filters]

        cat = torch.cat(pooled, dim=-1)
        # cat = [batch size, n filters * len(filter_sizes)]


        prediction = self.fc(cat)
        # prediction = [batch size, output dim]
        return prediction




Now, set hyperparameter values and initialize the model.

In [ ]:
vocab_size = len(vocab)
embedding_dim = 300
n_filters = 50
filter_sizes = [2, 3, 4]
batch_size = 25

output_dim = len(train_data.unique("label"))

model = CNN(
    vocab_size,
    embedding_dim,
    n_filters,
    filter_sizes,
    output_dim,
    pad_index,
)


def initialize_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv1d):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

model.apply(initialize_weights)

optimizer = optim.SGD(model.parameters(),weight_decay=0.0025)

criterion = nn.CrossEntropyLoss()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

model = model.to(device)
criterion = criterion.to(device)

Define the training and evaluation function

In [ ]:
def train(data_loader, model, criterion, optimizer, device):
    model.train()
    epoch_losses = []
    epoch_accs = []
    for batch in tqdm.tqdm(data_loader, desc="training..."):
        ids = batch["ids"].to(device)
        label = batch["label"].to(device)
        prediction = model(ids)
        loss = criterion(prediction, label)
        accuracy = get_accuracy(prediction, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        epoch_accs.append(accuracy.item())

    return np.mean(epoch_losses), np.mean(epoch_accs)

def evaluate(data_loader, model, criterion, device):
    model.eval()
    epoch_losses = []
    epoch_accs = []
    with torch.no_grad():
        for batch in tqdm.tqdm(data_loader, desc="evaluating..."):
            ids = batch["ids"].to(device)
            label = batch["label"].to(device)
            prediction = model(ids)
            loss = criterion(prediction, label)
            accuracy = get_accuracy(prediction, label)
            epoch_losses.append(loss.item())
            epoch_accs.append(accuracy.item())
    return np.mean(epoch_losses), np.mean(epoch_accs)

def get_accuracy(prediction, label):
    batch_size, _ = prediction.shape
    predicted_classes = prediction.argmax(dim=-1)
    correct_predictions = predicted_classes.eq(label).sum()
    accuracy = correct_predictions / batch_size
    return accuracy

Perform training. The total number of epochs is 50. The best model on the validation set is recorded.

In [ ]:
n_epochs = 50
best_valid_acc = 0

metrics = collections.defaultdict(list)

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

for epoch in range(n_epochs):
    train_loss, train_acc = train(
        train_data_loader, model, criterion, optimizer, device
    )
    valid_loss, valid_acc = evaluate(valid_data_loader, model, criterion, device)
    metrics["train_losses"].append(train_loss)
    metrics["train_accs"].append(train_acc)
    metrics["valid_losses"].append(valid_loss)
    metrics["valid_accs"].append(valid_acc)
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        torch.save(model.state_dict(), "cnn.pt")
    print(f"epoch: {epoch}")
    print(f"train_loss: {train_loss:.3f}, train_acc: {train_acc:.3f}")
    print(f"valid_loss: {valid_loss:.3f}, valid_acc: {valid_acc:.3f}")

Finish training... Let's load the best model and check the performance on the test set.

In [ ]:
model.load_state_dict(torch.load("cnn.pt"))

test_loss, test_acc = evaluate(test_data_loader, model, criterion, device)

print(f"test_loss: {test_loss:.3f}, test_acc: {test_acc:.3f}")

You can observe that the accuracy for both validation and test sets are low, less than 60%. However, the accuracy reported in the original paper for the "MR" dataset is 76.1% (CNN-rand).







===============================================================================================
# In what follows, you are required to complete two tasks to improve the performance.

===============================================================================================

## Task 1.

---


Make simple revisions to the above code based on your understanding of the original paper: [Convolutional Neural Networks for Sentence Classification](https://arxiv.org/pdf/1408.5882.pdf). **Here are all the steps you need to follow to complete this task.**
You are required to **insert** or **edit** directly on top of the given scripts above. After performing all the following edits, re-run the code to train the revised model and conduct evaluation. **Keep your training logs and evaluation result in this notebook.**





**Step 1. Edit the provided code to add dropout to the CNN model according to the paper's description. You need to locate the specific layer to apply dropout. The dropout rate should also follow the original paper.**

**Step 2. Edit the provided code to change the optimizer to the one specified in the paper.**


Explain what's the main difference between your selected optimizer and the one given above. Why would such replacement be beneficial for the final performance?

#########################
- Give your answer here:



#########################

**Step 3. Edit the provided code to change the pooling function to the one specified in the paper.**


**Step 4. Edit the provided code to change the number of filters, filter sizes and batch size to those specified in the paper.**







**Step 5. Re-run the revised code from the beginning. Conduct training and evaluation in the same way given in the original code. Keep the training log and evaluation result in this notebook.**

## Task 2.

---



Make component or architectural changes to the above model to further enhance the performance. In the following, you can find some suggested options. You are free to use other possible solutions.



*   Replace random word embeddings in the above model with pretrained word embeddings (Glove or word2vec).
*   Replace CNN with LSTM or BiLSTM.
*   Replace CNN with Transformers such as BERT.


**To complete this task, follow the questions below:**





**Question 1.   Clearly specify what component or architecture change you choose. Explain why you think it is helpful.**

#########################
- Give your answer here:



#########################

**Question 2.   Specify which part of the above model you plan to revise.**





#########################
- Give your answer here:



#########################



**Question 3.   Show the script of the revised model in a separate notebook. (Make a copy of the above code in a new notebook and edit the code correspondingly.) Re-run the code to train the new model and perform evaluation. Keep all the scripts you tried, the training logs and evaluation results.**